In [40]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, GroupKFold
from missforest import MissForest
import warnings

In [41]:
PATH_TO_FINAL_DATASET = "data_cleaned/final_dataset.parquet"

In [42]:
df = pd.read_parquet(PATH_TO_FINAL_DATASET, engine='pyarrow')

In [43]:
phq8_demographics = df.drop(columns=[col for col in df.columns if col.startswith('sleep_')])
phq8_demographics = phq8_demographics.drop(columns = ["timeInterval_ID"])

In [44]:
X = phq8_demographics.drop(columns=['phq8_score'])


In [45]:
demographic_cols = [
    'age',
    'gender_at_birth',
    'marital_status',
    'children',
    'ethnicity_1',
    'eduction_finish_age',
    'drop_school_1',
    'income_gbp',
    'accommodation',
    'mh_smoking',
    'audit_1',
    'depression_medication'
]

In [46]:
categorical_cols = ['marital_status', 'ethnicity_1', 'accommodation']


In [47]:
for col in categorical_cols:
    X[col] = X[col].cat.codes
    X[col] = X[col].replace(-1, np.nan)

In [48]:
ordinal_cols = ['income_gbp', 'mh_smoking', 'audit_1']

In [49]:
for col in ordinal_cols:
    X[col] = X[col].cat.codes
    X[col] = X[col].replace(-1, np.nan)

In [50]:
X['gender_at_birth'] = X['gender_at_birth'].map({'Female': 0, 'Male': 1})
X['children'] = X['children'].map({'No': 0, 'Yes': 1})
X['drop_school_1'] = X['drop_school_1'].map({'No': 0, 'Yes': 1})
X['depression_medication'] = X['depression_medication'].map({'No': 0, 'Yes': 1})

In [51]:
X['gender_at_birth'] = X['gender_at_birth'].astype(np.float64)  
X['depression_medication'] = X['depression_medication'].astype(np.float64)
X['children'] = X['children'].astype(np.float64)
X['drop_school_1'] = X['drop_school_1'].astype(np.float64)

In [52]:
demographics = X.groupby('participant_name')[demographic_cols].first()

In [53]:
imputer = MissForest(categorical = categorical_cols)

In [54]:
imputer.fit(demographics)

/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:333: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
100%|██████████| 5/5 [00:39<00:00,  7.88s/it]


In [55]:
demographics_imputed = imputer.transform(demographics)

/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:490: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:494: UserWarning: In version 4.2.3, estimator fitting process is moved to `fit` method. `MissForest` will now imputes unseen missing values with fitted estimators with `transform` method. To retain the old behaviour, use `fit_transform` to fit the whole unseen data instead.
  warnings.warn(f"In version {VERSION}, estimator fitting process "
100%|██████████| 5/5 [00:00<00:00, 49.13it/s]


In [56]:
type(demographics_imputed)

pandas.DataFrame

In [57]:
demographics_imputed.isnull().sum()

age                      0
marital_status           0
accommodation            0
mh_smoking               0
children                 0
eduction_finish_age      0
audit_1                  0
depression_medication    0
gender_at_birth          0
ethnicity_1              0
drop_school_1            0
income_gbp               0
dtype: int64

In [58]:
for col in demographics_imputed.columns:
    print(col)
    print(demographics_imputed[col].unique())
    print("\n")

age
[55.         56.         34.         62.         20.         26.
 58.         37.         38.         41.         52.         72.
 22.         23.         61.         32.         47.         63.
 46.         66.         51.         59.         40.         28.
 24.         53.         44.         21.         57.         54.
 29.         50.         64.         42.         35.         31.
 45.         30.         48.         27.         25.         69.
 19.         39.         65.         70.         67.         36.
 60.         49.         68.         43.         33.         71.
 18.         76.         50.29600036 75.        ]


marital_status
[2. 1. 5. 0. 4. 3.]


accommodation
[4. 2. 1. 0. 5. 3. 6. 7.]


mh_smoking
[2.         1.         0.         1.41059073]


children
[1.         0.         0.45595512 0.02910472]


eduction_finish_age
[27.         21.         19.         20.         24.         29.
 16.         18.         25.         22.         23.         17.
 28.         2

In [59]:
X_no_demo = X.drop(columns=demographic_cols)


In [60]:
demographics_imputed.head(5)

,age,marital_status,accommodation,mh_smoking,children,eduction_finish_age,audit_1,depression_medication,gender_at_birth,ethnicity_1,drop_school_1,income_gbp
participant_name,,,,,,,,,,,,
007751c5-d7ad-4bec-a58f-abf32500e2ae,55.0,2.0,4.0,2.0,1.0,27.0,0.0,1.000000,0.159003,0.0,0.046790,2.183756
00bf567f-30ef-4317-9c3d-f62a37bcce27,56.0,1.0,2.0,1.0,1.0,21.0,1.0,1.000000,0.000000,17.0,0.000000,2.115939
0104dfff-4dcd-48ff-b912-51362f098ed0,34.0,1.0,1.0,1.0,1.0,21.0,1.0,0.836573,0.000000,17.0,0.000000,2.180683
01318193-bb33-4c1b-b1bf-d316a014b5ad,62.0,5.0,0.0,2.0,1.0,19.0,3.0,1.000000,-0.256005,0.0,0.035308,2.038822
0157930f-10f7-4d5f-9575-a65efbd1d505,20.0,0.0,5.0,1.0,0.0,20.0,1.0,1.074364,0.000000,0.0,0.000000,0.000000


In [61]:
X_no_demo.merge(demographics_imputed, left_on='participant_name', right_index=True, how = 'left')

,participant_name,age,marital_status,accommodation,mh_smoking,children,eduction_finish_age,audit_1,depression_medication,gender_at_birth,ethnicity_1,drop_school_1,income_gbp
0,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,2.0,0.0,0.0,1.0,17.0,4.000000,1.065801,0.000000,0.0,0.000000,4.000000
1,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,2.0,0.0,0.0,1.0,17.0,4.000000,1.065801,0.000000,0.0,0.000000,4.000000
2,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,2.0,0.0,0.0,1.0,17.0,4.000000,1.065801,0.000000,0.0,0.000000,4.000000
3,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,2.0,0.0,0.0,1.0,17.0,4.000000,1.065801,0.000000,0.0,0.000000,4.000000
4,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,2.0,0.0,0.0,1.0,17.0,4.000000,1.065801,0.000000,0.0,0.000000,4.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8354,5f3217f1-c751-4fee-a8c4-21acc5153d58,64.0,0.0,1.0,0.0,1.0,23.0,1.898202,1.000000,0.000000,0.0,0.000000,3.000000
8355,5f3217f1-c751-4fee-a8c4-21acc5153d58,64.0,0.0,1.0,0.0,1.0,23.0,1.898202,1.000000,0.000000,0.0,0.000000,3.000000
8356,5f3217f1-c751-4fee-a8c4-21acc5153d58,64.0,0.0,1.0,0.0,1.0,23.0,1.898202,1.000000,0.000000,0.0,0.000000,3.000000
8357,6b52c798-bf6e-4512-af0d-98e39939fd37,40.0,4.0,4.0,2.0,1.0,22.0,0.000000,1.000000,-0.135824,0.0,0.015782,1.729068


In [ ]:
class ImputerMerger(BaseEstimator, TransformerMixin):
    """
    Single transformer that:
    1. Extracts unique demographic rows (one per participant)
    2. Imputes using MissForest (handles mixed data types automatically)
    3. Merges with physiological data
    """
    
    def __init__(self, demographic_cols, categorical_cols,
                 participant_id_col='participant_name'):
        self.demographic_cols = demographic_cols
        self.categorical_cols = categorical_cols 
        self.participant_id_col = participant_id_col
        
        # Will be fitted
        self.imputer_ = None
        
    def fit(self, X):
        """Fit MissForest on unique demographic rows"""
        # Extract unique demographic rows (one per participant)
        demo_unique = X.groupby(self.participant_id_col)[self.demographic_cols].first()
        
        # Create and fit MissForest imputer
        self.imputer_ = MissForest (
            categorical=self.categorical_cols
        )

        # Fit on demographic data
        self.imputer_.fit(demo_unique)
        
        return self
    
    def transform(self, X):
        """Impute demographics and merge back"""
        
        # Extract unique demographic rows with participant IDs
        demo_unique = X.groupby(self.participant_id_col)[self.demographic_cols].first()
        
        # Impute
        demo_imputed = self.imputer_.transform(demo_unique)
        
        # Drop original demographic columns from X
        X_no_demo = X.drop(columns=self.demographic_cols)
        
        # Merge imputed demographics back
        X_merged = X_no_demo.merge(
            demo_imputed,
            left_on=self.participant_id_col,
            right_index=True,
            how='left'
        )
        
        
        return X_merged

In [72]:
imputer_merger = MissForestImputerMerger(
    demographic_cols=demographic_cols,
    categorical_cols=categorical_cols,
)

In [73]:
imputer_merger.fit(X)

/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:333: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:40<00:00,  8.01s/it]


,demographic_cols,"['age', 'gender_at_birth', ...]"
,categorical_cols,"['marital_status', 'ethnicity_1', ...]"
,participant_id_col,'participant_name'


In [74]:
X_imputed = imputer_merger.transform(X)

/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:490: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:494: UserWarning: In version 4.2.3, estimator fitting process is moved to `fit` method. `MissForest` will now imputes unseen missing values with fitted estimators with `transform` method. To retain the old behaviour, use `fit_transform` to fit the whole unseen data instead.
  warnings.warn(f"In version {VERSION}, estimator fitting process "
100%|██████████| 5/5 [00:00<00:00, 48.53it/s]


In [75]:
imputed_demos = X_imputed.groupby('participant_name')[demographic_cols].first()

In [76]:
imputed_demos.isnull().sum()

age                      0
gender_at_birth          0
marital_status           0
children                 0
ethnicity_1              0
eduction_finish_age      0
drop_school_1            0
income_gbp               0
accommodation            0
mh_smoking               0
audit_1                  0
depression_medication    0
dtype: int64

In [77]:
for col in imputed_demos.columns:
    print(col)
    print(imputed_demos[col].unique())
    print("\n")

age
[55.         56.         34.         62.         20.         26.
 58.         37.         38.         41.         52.         72.
 22.         23.         61.         32.         47.         63.
 46.         66.         51.         59.         40.         28.
 24.         53.         44.         21.         57.         54.
 29.         50.         64.         42.         35.         31.
 45.         30.         48.         27.         25.         69.
 19.         39.         65.         70.         67.         36.
 60.         49.         68.         43.         33.         71.
 18.         76.         50.29600036 75.        ]


gender_at_birth
[ 1.59003043e-01  0.00000000e+00 -2.56004817e-01  1.00000000e+00
  2.04654993e-01 -8.70531555e-02  7.93779455e-02  8.76757988e-03
  3.30895789e-01  1.48071683e-01  1.10506914e-01  1.97949486e-01
  3.59546398e-02 -9.00747431e-02 -5.66295110e-02 -8.90523063e-02
 -4.95088873e-02  8.82787068e-02  3.98787270e-01  8.44193143e-02
  7.22114051e-03 -

In [70]:
X_imputed.isnull().sum()

participant_name         0
age                      0
marital_status           0
accommodation            0
mh_smoking               0
children                 0
eduction_finish_age      0
audit_1                  0
depression_medication    0
gender_at_birth          0
ethnicity_1              0
drop_school_1            0
income_gbp               0
dtype: int64